# Suite2 Project7 Churn

**Dataset:** Telco Customer Churn (Kaggle, 7,043 rows)

---
Run each cell in order. All outputs are generated from real data.

In [ ]:
# Setup: imports and configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.dpi'] = 120

# Helper: load dataset
def load_data(name):
    return pd.read_csv(f'https://raw.githubusercontent.com/ulink-ibcs/A4_ML_Projects/main/datasets/{name}')

print("✅ Setup complete!")

In [ ]:
"""Project 7: Customer Churn — Classification"""
import sys, os; sys.path.insert(0, os.path.dirname(__file__))

df = load_csv('telco_churn.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'suite2_project7_outputs'); os.makedirs(OUTPUT_DIR, exist_ok=True)

                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)

## PROJECT 7: CUSTOMER CHURN — CLASSIFICATION IN PRACTICE

In [ ]:
print(f"Dataset: Telco Customer Churn ({df.shape[0]:,} rows × {df.shape[1]} cols)")
print(f"\nColumns: {', '.join(df.columns)}")

# ── 1. Data Preparation ──

## 1. Data Preparation & Cleaning

In [ ]:
# Clean TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print(f"Missing TotalCharges: {df['TotalCharges'].isnull().sum()}")
df = df.dropna(subset=['TotalCharges'])

# Convert target
df['Churn_Label'] = (df['Churn'] == 'Yes').astype(int)
print(f"Churn distribution:\n{df['Churn'].value_counts().to_string()}")
print(f"Churn rate: {df['Churn_Label'].mean():.1%}")

# Feature engineering
cat_cols = df.select_dtypes(include=['object']).columns.drop(['customerID', 'Churn'])
print(f"\nCategorical features ({len(cat_cols)}): {list(cat_cols)}")

# Encode categoricals
le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col + '_enc'] = le.fit_transform(df[col].astype(str))
    le_dict[col] = dict(zip(le.classes_, le.transform(le.classes_)))

num_features = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_features = [c + '_enc' for c in cat_cols]
features = num_features + cat_features

X = df[features]
y = df['Churn_Label']

print(f"\nFeature matrix: {X.shape[0]:,} samples × {X.shape[1]} features")
print(f"Numeric features: {num_features}")
print(f"Encoded categorical: {cat_features}")

# ── 2. Train/Test Split ──

## 2. Train/Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Test churn rate: {y_test.mean():.1%}")

# ── 3. K-NN Classifier ──

## 3. K-Nearest Neighbours

In [ ]:
for k in [3, 5, 7, 11, 15]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    print(f"  k={k:2d}: Accuracy={acc:.4f}")

best_knn = KNeighborsClassifier(n_neighbors=7)
best_knn.fit(X_train_scaled, y_train)
y_pred_knn = best_knn.predict(X_test_scaled)
y_prob_knn = best_knn.predict_proba(X_test_scaled)[:, 1]

print(f"\n=== K-NN (k=7) Metrics ===")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_knn):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_knn):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_knn):.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_pred_knn):.4f}")
print(f"  ROC AUC:   {roc_auc_score(y_test, y_prob_knn):.4f}")

# ── 4. Decision Tree ──

## 4. Decision Tree Classifier

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42, min_samples_leaf=50)
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

print(f"=== Decision Tree (max_depth=5) Metrics ===")
print(f"  Accuracy:  {accuracy_score(y_test, y_pred_dt):.4f}")
print(f"  Precision: {precision_score(y_test, y_pred_dt):.4f}")
print(f"  Recall:    {recall_score(y_test, y_pred_dt):.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_pred_dt):.4f}")
print(f"  ROC AUC:   {roc_auc_score(y_test, y_prob_dt):.4f}")

# Feature importance
fi = pd.DataFrame({'Feature': features, 'Importance': dt.feature_importances_}).sort_values('Importance', ascending=False)
print(f"\nTop 10 Features by Importance:")
print(fi.head(10).to_string(index=False))

# ── 5. ROC Curve ──

## 5. ROC Curve Analysis

In [ ]:
fpr_knn, tpr_knn, _ = roc_curve(y_test, y_prob_knn)
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)

fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(fpr_knn, tpr_knn, 'b-', linewidth=2, label=f'K-NN (AUC = {roc_auc_score(y_test, y_prob_knn):.3f})')
ax.plot(fpr_dt, tpr_dt, 'r-', linewidth=2, label=f'Decision Tree (AUC = {roc_auc_score(y_test, y_prob_dt):.3f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — Churn Prediction', fontsize=14)
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
# 'p7_roc_curve')
plt.show()

# ── 6. Confusion Matrices ──
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_knn, ax=axes[0], cmap='Blues', 
                                         display_labels=['Stayed', 'Churned'])
axes[0].set_title('K-NN Confusion Matrix', fontsize=13)
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_dt, ax=axes[1], cmap='Blues',
                                         display_labels=['Stayed', 'Churned'])
axes[1].set_title('Decision Tree Confusion Matrix', fontsize=13)
plt.tight_layout()
# 'p7_confusion_matrices')
plt.show()
print("[Saved] p7_*.png")

# ── Export ──
results = {
    'dataset_shape': list(df.shape),
    'churn_rate': float(df['Churn_Label'].mean()),
    'knn_accuracy': float(accuracy_score(y_test, y_pred_knn)),
    'knn_precision': float(precision_score(y_test, y_pred_knn)),
    'knn_recall': float(recall_score(y_test, y_pred_knn)),
    'knn_f1': float(f1_score(y_test, y_pred_knn)),
    'knn_auc': float(roc_auc_score(y_test, y_prob_knn)),
    'dt_accuracy': float(accuracy_score(y_test, y_pred_dt)),
    'dt_precision': float(precision_score(y_test, y_pred_dt)),
    'dt_recall': float(recall_score(y_test, y_pred_dt)),
    'dt_f1': float(f1_score(y_test, y_pred_dt)),
    'dt_auc': float(roc_auc_score(y_test, y_prob_dt)),
    'top_features': fi.head(10).to_dict('records'),
}
    json.dump(results, f, indent=2, default=str)

print("Done.")